In [1]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from gwgraphs import *

In [2]:
def read_graph_data():
  # Read the adjacency matrix (edges)
  with open("IMDB-MULTI_A.txt", "r") as f:
      edges = [tuple(map(int, line.split(','))) for line in f]

  # Read graph indicators
  with open("IMDB-MULTI_graph_indicator.txt", "r") as f:
      graph_indicators = [int(line.strip()) for line in f if line.strip()]

  # Read graph labels
  with open("IMDB-MULTI_graph_labels.txt", "r") as f:
      graph_labels = [int(line.strip()) for line in f]

  return edges, graph_indicators, graph_labels


def construct_networkx_graph(edges, graph_indicators, graph_labels):
  graphs = {}
  for node_id, graph_id in enumerate(graph_indicators, start=1):
      if graph_id not in graphs:
          graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

  for edge in edges:
      node1, node2 = edge
      graph_id = graph_indicators[
          node1 - 1]  # Determine which graph the edge belongs to
      graphs[graph_id].add_edge(node1, node2)

  return graphs


def compute_averages(graphs):
  # Initialize variables
  total_nodes = 0
  class_nodes = {}
  class_counts = {}
  # Iterate through graphs to compute totals
  for graph_id, graph in graphs.items():
      num_nodes = graph.number_of_nodes()
      total_nodes += num_nodes

      # Get the class label for the current graph
      graph_label = graph.graph['label']

      if graph_label not in class_nodes:
          class_nodes[graph_label] = 0
          class_counts[graph_label] = 0

      class_nodes[graph_label] += num_nodes
      class_counts[graph_label] += 1
  # Calculate total average
  total_avg = round(total_nodes / len(graphs))

  # Calculate class-based averages
  class_averages = {label: round(class_nodes[label] / count)
                    for label, count in class_counts.items()}

  return total_avg, class_averages


def compute_distance_matrix(graph):
  num_nodes = len(graph.nodes)
  distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

  # Get the correct node indices to be used for indexing the matrix
  node_list = list(graph.nodes())
  node_index = {node: i for i, node in enumerate(node_list)}

  for node in graph.nodes:
      lengths = nx.shortest_path_length(graph, source=node, weight="weight")

      for target, length in lengths.items():
          if target in node_index:
              distance_matrix[node_index[node], node_index[target]] = length

  return distance_matrix

In [3]:
edges, graph_indicators, graph_labels = read_graph_data()
graphs = construct_networkx_graph(edges, graph_indicators, graph_labels)
for graph_id, graph in graphs.items():
    print(
        f"Graph ID: {graph_id}, number of nodes: {graph.number_of_nodes()}, label: {graph.graph['label']}"
        )
total_avg, class_averages = compute_averages(graphs)
print(f"Average number of nodes in entire dataset: {total_avg}")
print("Average number of nodes for each class:")
for class_label, avg_nodes in class_averages.items():
        print(f"  Class {class_label}: {avg_nodes}")

Graph ID: 1, number of nodes: 7, label: 1
Graph ID: 2, number of nodes: 13, label: 1
Graph ID: 3, number of nodes: 7, label: 1
Graph ID: 4, number of nodes: 7, label: 1
Graph ID: 5, number of nodes: 10, label: 1
Graph ID: 6, number of nodes: 8, label: 1
Graph ID: 7, number of nodes: 7, label: 1
Graph ID: 8, number of nodes: 8, label: 1
Graph ID: 9, number of nodes: 13, label: 1
Graph ID: 10, number of nodes: 22, label: 1
Graph ID: 11, number of nodes: 8, label: 1
Graph ID: 12, number of nodes: 17, label: 1
Graph ID: 13, number of nodes: 8, label: 1
Graph ID: 14, number of nodes: 22, label: 1
Graph ID: 15, number of nodes: 7, label: 1
Graph ID: 16, number of nodes: 16, label: 1
Graph ID: 17, number of nodes: 14, label: 1
Graph ID: 18, number of nodes: 7, label: 1
Graph ID: 19, number of nodes: 9, label: 1
Graph ID: 20, number of nodes: 8, label: 1
Graph ID: 21, number of nodes: 12, label: 1
Graph ID: 22, number of nodes: 24, label: 1
Graph ID: 23, number of nodes: 9, label: 1
Graph ID: 

In [4]:
graphs_list = []
labels_list = []
for graph_id, graph in graphs.items():
    # Add the graph object to the list
    graphs_list.append(graph)

    # Extract the label and add to the label list
    label = graph.graph['label']
    # Convert label 0 to -1
    if label == 0:
        label = -1
    labels_list.append(label)

# Convert graphs into pairwise distance matrices
distance_matrices = [compute_distance_matrix(graph) for graph in graphs_list]

# Determine number of elements of each class
print("Elements of Class 1: ", labels_list.count(1))
print("Elements of Class 2: ", labels_list.count(2))
print("Elements of Class 3: ", labels_list.count(3))

Elements of Class 1:  500
Elements of Class 2:  500
Elements of Class 3:  500


In [5]:
def gw_knn_multi(distance_matrices,
                  labels,
                  num_folds,
                  k_list,
                  method,
                  seed=42,
                  log_knn=False):
    """Function to perform k-nearest neighbors classification using Gromov-Wasserstein distances for multiclass classification.
    Parameters:
    - distance_matrices: List of distance matrices.
    - labels: List of labels corresponding to the distance matrices.
    - num_folds: Number of folds for cross-validation.
    - k_list: List of k values to test.
    - method: Cross-validation method, either 'kfold' or 'stratk'.
    - seed: Random seed for reproducibility. Defaults to 42.
    - log_knn: Boolean flag to log k-NN details. Defaults to False.
    
    Returns:
    - class_acc_dict: Dictionary storing accuracies for each k.
    - by_class_acc_dict: Dictionary storing accuracies by class for each k.
    - f1_score_dict: Dictionary storing F1 scores for each k.
    - best_k_info: Dictionary storing information about the best k value.
    """
    class_acc_dict = {
        k: []
        for k in k_list
    }  # dictionary to store accuracies for each k
    by_class_acc_dict = {
        k: {
            1: [],
            2: [],
            3: []
        }
        for k in k_list
    }  # store accuracies by class for each k
    f1_score_dict = {
        k: []
        for k in k_list
    }  # dictionary to store F1 scores for each k
    assert method in ["kfold", "stratk"]
    if method == "kfold":
        skf = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
        print("k-fold CV selected")
    else:
        skf = StratifiedKFold(n_splits=num_folds,
                              shuffle=True,
                              random_state=seed)
        print("Stratified k-fold CV selected")
    start = time.time()

    for i, (train_index,
            test_index) in enumerate(skf.split(distance_matrices, labels)):
        print(f"Fold {i}:")
        if log_knn:
            print(f"  Train: index={train_index}")
            print(f"  Test:  index={test_index}")

        X_train = [distance_matrices[idx] for idx in train_index]
        y_train = [labels[idx] for idx in train_index]
        X_test = [distance_matrices[idx] for idx in test_index]
        y_test = [labels[idx] for idx in test_index]

        # Compute all pairwise distances once per fold
        all_distances = []
        for test_graph, ground_truth_label in zip(X_test, y_test):
            distances = []
            for train_graph, train_label in zip(X_train, y_train):
                gw, log = ot.gromov.gromov_wasserstein(test_graph,
                                                       train_graph,
                                                       loss_fun='square_loss',
                                                       log=True,
                                                       random_seed=seed)
                distances.append((log['gw_dist'], train_label))
            distances.sort(key=lambda x: x[0])  # Sort based on distances
            all_distances.append((distances, ground_truth_label))

        # For each k value, perform classification
        for k in k_list:
            correct_count = 0
            class_correct = {1: 0, 2: 0, 3: 0}
            class_total = {1: 0, 2: 0, 3: 0}
            y_true = []
            y_pred = []

            for distances, ground_truth_label in all_distances:
                # Take the k nearest neighbors
                k_nearest_neighbors = distances[:k]
                # Determine the most common label
                labels_counts = {}
                for _, label in k_nearest_neighbors:
                    labels_counts[label] = labels_counts.get(label, 0) + 1
                predicted_label = max(labels_counts, key=labels_counts.get)

                y_true.append(ground_truth_label)
                y_pred.append(predicted_label)

                if predicted_label == ground_truth_label:
                    correct_count += 1
                    class_correct[ground_truth_label] += 1
                class_total[ground_truth_label] += 1

            # Calculate overall and by-class accuracies
            classification_accuracy = (correct_count / len(X_test)) * 100
            class_acc_dict[k].append(classification_accuracy)

            # Calculate F1 score with appropriate averaging
            if method == "kfold":
                f1 = f1_score(y_true, y_pred, average='macro') * 100
            else:  # stratk
                f1 = f1_score(y_true, y_pred, average='weighted') * 100
            f1_score_dict[k].append(f1)

            # Calculate and store by-class accuracies
            for class_label in [1, 2, 3]:
                if class_total[class_label] > 0:
                    class_accuracy = (class_correct[class_label] /
                                      class_total[class_label]) * 100
                    by_class_acc_dict[k][class_label].append(class_accuracy)
                    print(
                        f"k={k}, Class {class_label} Accuracy: {class_accuracy}%, Fold {i}"
                    )

            print(
                f"k={k}, Overall Classification Accuracy: {classification_accuracy}%, Fold {i}"
            )
            average_type = "macro" if method == "kfold" else "weighted"
            print(f"k={k}, F1 Score ({average_type} avg): {f1:.2f}%, Fold {i}")

    end = time.time()
    print("Total runtime, seconds:", end - start)

    # Print accuracy stats for each k and print best overall k
    best_k = k_list[0]
    best_acc = 0
    best_f1 = 0
    k_avg_accuracies = {}
    k_avg_f1_scores = {}
    k_std_accuracies = {}
    k_std_f1_scores = {}

    average_type = "macro" if method == "kfold" else "weighted"

    for k in k_list:
        avg_acc = sum(class_acc_dict[k]) / len(class_acc_dict[k])
        avg_f1 = sum(f1_score_dict[k]) / len(f1_score_dict[k])

        # Calculate standard deviations
        std_acc = np.std(class_acc_dict[k])
        std_f1 = np.std(f1_score_dict[k])
        std_class_1 = np.std(by_class_acc_dict[k][1])
        std_class_2 = np.std(by_class_acc_dict[k][2])
        std_class_3 = np.std(by_class_acc_dict[k][3])

        k_avg_accuracies[k] = {
            'overall':
            avg_acc,
            'class_1':
            sum(by_class_acc_dict[k][1]) / len(by_class_acc_dict[k][1]),
            'class_2':
            sum(by_class_acc_dict[k][2]) / len(by_class_acc_dict[k][2]),
            'class_3':
            sum(by_class_acc_dict[k][3]) / len(by_class_acc_dict[k][3])
        }
        k_avg_f1_scores[k] = avg_f1
        k_std_accuracies[k] = {
            'overall': std_acc,
            'class_1': std_class_1,
            'class_2': std_class_2,
            'class_3': std_class_3
        }
        k_std_f1_scores[k] = std_f1

        print(f"\nk={k} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"k={k} Average F1 Score ({average_type} avg): {avg_f1:.2f}%")
        for class_label in [1, 2, 3]:
            print(
                f"k={k} Average Class {class_label} Accuracy: {k_avg_accuracies[k][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_acc:
            best_acc = avg_acc
            best_k = k

        if avg_f1 > best_f1:
            best_f1 = avg_f1

    print(f"\nBest performance achieved with k={best_k}:")
    print(
        f"Overall Accuracy: {k_avg_accuracies[best_k]['overall']:.2f}% ± {k_std_accuracies[best_k]['overall']:.2f}%"
    )
    print(
        f"F1 Score ({average_type} avg): {k_avg_f1_scores[best_k]:.2f}% ± {k_std_f1_scores[best_k]:.2f}%"
    )
    print(
        f"Class 1 Accuracy: {k_avg_accuracies[best_k]['class_1']:.2f}% ± {k_std_accuracies[best_k]['class_1']:.2f}%"
    )
    print(
        f"Class 2 Accuracy: {k_avg_accuracies[best_k]['class_2']:.2f}% ± {k_std_accuracies[best_k]['class_2']:.2f}%"
    )
    print(
        f"Class 3 Accuracy: {k_avg_accuracies[best_k]['class_3']:.2f}% ± {k_std_accuracies[best_k]['class_3']:.2f}%"
    )

    # Prepare best k information to return
    best_k_info = {
        'best_k': best_k,
        'best_overall_accuracy': k_avg_accuracies[best_k]['overall'],
        'best_f1_score': k_avg_f1_scores[best_k],
        'best_class_1_accuracy': k_avg_accuracies[best_k]['class_1'],
        'best_class_2_accuracy': k_avg_accuracies[best_k]['class_2'],
        'best_class_3_accuracy': k_avg_accuracies[best_k]['class_3']
    }

    return class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info

In [6]:
num_folds = 10
k_list = [1,2,3,4,5,6,7,8,9,10]

In [8]:
accuracies, by_class_accuracies, f1_scores, best_k_info = gw_knn_multi(distance_matrices, labels_list, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 67.24137931034483%, Fold 0
k=1, Class 2 Accuracy: 17.073170731707318%, Fold 0
k=1, Class 3 Accuracy: 11.76470588235294%, Fold 0
k=1, Overall Classification Accuracy: 34.66666666666667%, Fold 0
k=1, F1 Score (macro avg): 28.61%, Fold 0
k=2, Class 1 Accuracy: 67.24137931034483%, Fold 0
k=2, Class 2 Accuracy: 17.073170731707318%, Fold 0
k=2, Class 3 Accuracy: 11.76470588235294%, Fold 0
k=2, Overall Classification Accuracy: 34.66666666666667%, Fold 0
k=2, F1 Score (macro avg): 28.61%, Fold 0
k=3, Class 1 Accuracy: 68.96551724137932%, Fold 0
k=3, Class 2 Accuracy: 21.951219512195124%, Fold 0
k=3, Class 3 Accuracy: 13.725490196078432%, Fold 0
k=3, Overall Classification Accuracy: 37.333333333333336%, Fold 0
k=3, F1 Score (macro avg): 31.91%, Fold 0
k=4, Class 1 Accuracy: 67.24137931034483%, Fold 0
k=4, Class 2 Accuracy: 24.390243902439025%, Fold 0
k=4, Class 3 Accuracy: 13.725490196078432%, Fold 0
k=4, Overall Classification Accuracy: 37.3333

In [9]:
# ATTEMPT 2: Try adjacency matrices
def compute_adjacency_matrix(graph):
    """
    Computes the adjacency matrix for a given graph.
    
    Args:
        graph: A NetworkX graph.
    
    Returns:
        A numpy array representing the adjacency matrix where the value 
        at (i, j) denotes the presence of an edge between node i and node j.
    """
    # Initialize a matrix of zeros
    num_nodes = len(graph.nodes)
    adjacency_matrix = np.zeros((num_nodes, num_nodes))
    
    # Populate the adjacency matrix
    for i, node_i in enumerate(graph.nodes()):
        for j, node_j in enumerate(graph.nodes()):
            if graph.has_edge(node_i, node_j):
                adjacency_matrix[i, j] = 1.0
    return adjacency_matrix

In [10]:
adj_matrices = [compute_adjacency_matrix(graph) for graph in graphs_list]
accuraciesadj, by_class_accuraciesadj, f1_scoresadj, best_k_infoadj = gw_knn_multi(adj_matrices, labels_list, num_folds, k_list=k_list, method="kfold")

k-fold CV selected
Fold 0:
k=1, Class 1 Accuracy: 70.6896551724138%, Fold 0
k=1, Class 2 Accuracy: 21.951219512195124%, Fold 0
k=1, Class 3 Accuracy: 13.725490196078432%, Fold 0
k=1, Overall Classification Accuracy: 38.0%, Fold 0
k=1, F1 Score (macro avg): 32.31%, Fold 0
k=2, Class 1 Accuracy: 70.6896551724138%, Fold 0
k=2, Class 2 Accuracy: 21.951219512195124%, Fold 0
k=2, Class 3 Accuracy: 13.725490196078432%, Fold 0
k=2, Overall Classification Accuracy: 38.0%, Fold 0
k=2, F1 Score (macro avg): 32.31%, Fold 0
k=3, Class 1 Accuracy: 68.96551724137932%, Fold 0
k=3, Class 2 Accuracy: 17.073170731707318%, Fold 0
k=3, Class 3 Accuracy: 9.803921568627452%, Fold 0
k=3, Overall Classification Accuracy: 34.66666666666667%, Fold 0
k=3, F1 Score (macro avg): 27.93%, Fold 0
k=4, Class 1 Accuracy: 68.96551724137932%, Fold 0
k=4, Class 2 Accuracy: 19.51219512195122%, Fold 0
k=4, Class 3 Accuracy: 11.76470588235294%, Fold 0
k=4, Overall Classification Accuracy: 36.0%, Fold 0
k=4, F1 Score (macro av

In [11]:
"""Multiclass gLGW-kNN"""
def glgw_knn_multi(lgw_matrix,
                   labels,
                   train_index,
                   test_index,
                   k_list,
                   method,
                   log_knn=False):
    """
    Computes the k-nearest neighbors for a given graph using the LGW distance for multiclass classification.
    Parameters:
    - lgw_matrix: The LGW distance matrix containing all the pairwise LGW distances between graphs.
    - labels: The labels of the graphs.
    - train_index: The indices of the graphs from the training set.
    - test_index: The indices of the graphs from the test set.
    - k_list: List of k values to evaluate.
    - method: Cross-validation method, either 'kfold' or 'stratk'.
    - log_knn: Whether to print the classification accuracy log. Defaults to False.
    
    Returns:
    - class_acc_dict: Dictionary storing accuracies for each k.
    - by_class_acc_dict: Dictionary storing accuracies by class for each k.
    - f1_score_dict: Dictionary storing F1 scores for each k.
    """
    class_acc_dict = {k: [] for k in k_list}
    by_class_acc_dict = {k: {1: [], 2: [], 3: []} for k in k_list}
    f1_score_dict = {k: [] for k in k_list}

    X_train, y_train = lgw_matrix[train_index], [
        labels[idx] for idx in train_index
    ]
    X_test, y_test = lgw_matrix[test_index], [
        labels[idx] for idx in test_index
    ]

    # Initialize counters for each k
    correct_counts = {k: 0 for k in k_list}
    class_correct = {k: {1: 0, 2: 0, 3: 0} for k in k_list}
    class_total = {k: {1: 0, 2: 0, 3: 0} for k in k_list}
    predictions = {k: [] for k in k_list}

    for test_idx, ground_truth_label in zip(test_index, y_test):
        distances = [(lgw_matrix[test_idx, idx], y_train[j])
                     for j, idx in enumerate(train_index)]
        distances.sort(key=lambda x: x[0])

        for k in k_list:
            k_nearest_neighbors = distances[:k]
            labels_counts = {}
            for _, label in k_nearest_neighbors:
                labels_counts[label] = labels_counts.get(label, 0) + 1
            predicted_label = max(labels_counts, key=labels_counts.get)

            predictions[k].append(predicted_label)

            if predicted_label == ground_truth_label:
                correct_counts[k] += 1
                class_correct[k][ground_truth_label] += 1
            class_total[k][ground_truth_label] += 1

    # Calculate accuracies and F1 scores after all predictions
    average_type = "macro" if method == "kfold" else "weighted"

    for k in k_list:
        # Overall accuracy
        classification_accuracy = (correct_counts[k] / len(test_index)) * 100
        class_acc_dict[k].append(classification_accuracy)

        # Calculate F1 score with appropriate averaging
        if method == "kfold":
            f1 = f1_score(y_test, predictions[k], average='macro') * 100
        else:  # stratk
            f1 = f1_score(y_test, predictions[k], average='weighted') * 100
        f1_score_dict[k].append(f1)

        # By-class accuracies
        for class_label in [1, 2, 3]:
            if class_total[k][class_label] > 0:
                class_accuracy = (class_correct[k][class_label] /
                                  class_total[k][class_label]) * 100
                by_class_acc_dict[k][class_label].append(class_accuracy)
                if log_knn:
                    print(
                        f"k={k}, Class {class_label} Accuracy: {class_accuracy}%"
                    )

        if log_knn:
            print(
                f"k={k}, Overall Classification Accuracy: {classification_accuracy}%"
            )
            print(f"k={k}, F1 Score ({average_type} avg): {f1:.2f}%")

    return class_acc_dict, by_class_acc_dict, f1_score_dict


def k_folds_glgw_knn_multi(X, y, k_bary, Ms, heights, num_folds, seed, k_knn_list,
                           method):
    """
    Implements k-fold cross-validation in conjunction with the gLGW-kNN algorithm for multiclass classification.
    Parameters:
    - X: The input data.
    - y: The labels of the data.
    - k_bary: The number of points to use for the reference barycenter in the LGW computation.
    - Ms: The list of similarity matrices (a matrix of measures).
    - heights: The distribution/weights in the target space.
    - num_folds: The number of folds to use for cross-validation.
    - seed: The seed for the random number generator.
    - k_knn_list: The list of k values to use for k-fold cross-validation.
    - method: Choice of k-fold cross validation (kfold) or stratified k-fold cross validation (stratk).
    
    Returns:
    - class_acc_dict: Dictionary storing accuracies for each k.
    - by_class_acc_dict: Dictionary storing accuracies by class for each k.
    - f1_score_dict: Dictionary storing F1 scores for each k.
    - best_k_info: Dictionary storing information about the best k value.
    """
    np.random.seed(seed)
    assert method in ["kfold", "stratk"]
    if method == "kfold":
        k_folds = KFold(n_splits=num_folds, shuffle=True, random_state=seed)
        print("k-fold CV selected")
    else:
        k_folds = StratifiedKFold(n_splits=num_folds,
                                  shuffle=True,
                                  random_state=seed)
        print("Stratified k-fold CV selected")

    # Initialize storage for all metrics
    class_acc_dict = {k: [] for k in k_knn_list}
    by_class_acc_dict = {k: {1: [], 2: [], 3: []} for k in k_knn_list}
    f1_score_dict = {k: [] for k in k_knn_list}

    classes = [1, 2, 3]
    fold_num = 0

    for train_index, test_index in k_folds.split(X, y):
        print(f"Fold {fold_num}:")
        X_train, X_test = [Ms[idx] for idx in train_index
                           ], [Ms[idx] for idx in test_index]
        y_train, y_test = np.array([y[idx] for idx in train_index]), np.array(
            [y[idx] for idx in test_index])
        # Compute reference barycenter for the training set
        idx_bary = []
        for i in classes:
            # Find the indices of instances with label i
            indices = np.where(y_train == i)[0]
            if indices.size > 0:
                idx_bary.extend(
                    np.random.choice(train_index[indices],
                                     size=min(5, len(indices)),
                                     replace=False))
        bary_start_time = time.time()
        # Initialize reference matrix with k_bary size
        M_ref = np.zeros((k_bary, k_bary))
        selected_Ms = [Ms[i] for i in idx_bary]
        selected_heights = [heights[i] for i in idx_bary]
        lambdas = ot.unif(len(idx_bary))
        M_ref = ot.gromov.gromov_barycenters(k_bary,
                                             Cs=selected_Ms,
                                             ps=selected_heights,
                                             p=ot.unif(k_bary),
                                             lambdas=lambdas,
                                             loss_fun='square_loss',
                                             max_iter=200,
                                             tol=1e-12,
                                             random_state=seed)
        bary_computation_time = time.time() - bary_start_time
        print(
            f"Time taken for reference barycenter computation: {bary_computation_time:.4f} seconds"
        )
        height_ref = ot.unif(k_bary)
        lgw_matrix, lgw_time = lgw_procedure(M_ref,
                                             height_ref,
                                             None,
                                             Ms,
                                             heights,
                                             mode="graph")

        print(f"Time taken for LGW computation: {lgw_time:.4f} seconds")

        # Evaluate for each k in k_knn_list
        fold_class_acc, fold_by_class_acc, fold_f1_scores = glgw_knn_multi(
            lgw_matrix,
            y,
            train_index,
            test_index,
            k_knn_list,
            method,
            log_knn=True)

        # Accumulate results
        for k in k_knn_list:
            class_acc_dict[k].extend(fold_class_acc[k])
            f1_score_dict[k].extend(fold_f1_scores[k])
            for class_label in [1, 2, 3]:
                by_class_acc_dict[k][class_label].extend(
                    fold_by_class_acc[k][class_label])

        fold_num += 1

    # Find best k and print stats
    best_k = k_knn_list[0]
    best_acc = 0
    best_f1 = 0
    k_avg_accuracies = {}
    k_avg_f1_scores = {}
    k_std_accuracies = {}
    k_std_f1_scores = {}

    average_type = "macro" if method == "kfold" else "weighted"

    for k in k_knn_list:
        avg_acc = sum(class_acc_dict[k]) / len(class_acc_dict[k])
        avg_f1 = sum(f1_score_dict[k]) / len(f1_score_dict[k])

        # Calculate standard deviations
        std_acc = np.std(class_acc_dict[k])
        std_f1 = np.std(f1_score_dict[k])
        std_class_1 = np.std(by_class_acc_dict[k][1])
        std_class_2 = np.std(by_class_acc_dict[k][2])
        std_class_3 = np.std(by_class_acc_dict[k][3])

        k_avg_accuracies[k] = {
            'overall':
            avg_acc,
            'class_1':
            sum(by_class_acc_dict[k][1]) / len(by_class_acc_dict[k][1]),
            'class_2':
            sum(by_class_acc_dict[k][2]) / len(by_class_acc_dict[k][2]),
            'class_3':
            sum(by_class_acc_dict[k][3]) / len(by_class_acc_dict[k][3])
        }
        k_avg_f1_scores[k] = avg_f1

        k_std_accuracies[k] = {
            'overall': std_acc,
            'class_1': std_class_1,
            'class_2': std_class_2,
            'class_3': std_class_3
        }
        k_std_f1_scores[k] = std_f1

        print(f"\nk={k} Average Overall Accuracy: {avg_acc:.2f}%")
        print(f"k={k} Average F1 Score ({average_type} avg): {avg_f1:.2f}%")
        for class_label in [1, 2, 3]:
            print(
                f"k={k} Average Class {class_label} Accuracy: {k_avg_accuracies[k][f'class_{class_label}']:.2f}%"
            )

        if avg_acc > best_acc:
            best_acc = avg_acc
            best_k = k

        if avg_f1 > best_f1:
            best_f1 = avg_f1

    print(f"\nBest performance achieved with k={best_k}:")
    print(
        f"Overall Accuracy: {k_avg_accuracies[best_k]['overall']:.2f}% ± {k_std_accuracies[best_k]['overall']:.2f}%"
    )
    print(
        f"F1 Score ({average_type} avg): {k_avg_f1_scores[best_k]:.2f}% ± {k_std_f1_scores[best_k]:.2f}%"
    )
    print(
        f"Class 1 Accuracy: {k_avg_accuracies[best_k]['class_1']:.2f}% ± {k_std_accuracies[best_k]['class_1']:.2f}%"
    )
    print(
        f"Class 2 Accuracy: {k_avg_accuracies[best_k]['class_2']:.2f}% ± {k_std_accuracies[best_k]['class_2']:.2f}%"
    )
    print(
        f"Class 3 Accuracy: {k_avg_accuracies[best_k]['class_3']:.2f}% ± {k_std_accuracies[best_k]['class_3']:.2f}%"
    )

    # Prepare best k information to return
    best_k_info = {
        'best_k': best_k,
        'best_overall_accuracy': k_avg_accuracies[best_k]['overall'],
        'best_f1_score': k_avg_f1_scores[best_k],
        'best_class_1_accuracy': k_avg_accuracies[best_k]['class_1'],
        'best_class_2_accuracy': k_avg_accuracies[best_k]['class_2'],
        'best_class_3_accuracy': k_avg_accuracies[best_k]['class_3']
    }

    return class_acc_dict, by_class_acc_dict, f1_score_dict, best_k_info

In [13]:
# Normalize the matrices
Ms = []
for matrix in distance_matrices:
    Ms.append(matrix / np.max(matrix))

k_bary = 13  # use overall dataset average
heights = [
    np.full(len(graph), 1 / len(graph)) for graph in graphs_list
]
num_folds = 10

accuracies2, by_class_accuracies2, f1_scores2, best_k_info2 = k_folds_glgw_knn_multi(distance_matrices,
            labels_list,
            k_bary,
            Ms,
            heights,
            num_folds,
            seed=42,
            k_knn_list=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            method="kfold")

k-fold CV selected
Fold 0:
Time taken for reference barycenter computation: 0.8423 seconds
Time taken for LGW computation: 13.0648 seconds
k=1, Class 1 Accuracy: 72.41379310344827%
k=1, Class 2 Accuracy: 14.634146341463413%
k=1, Class 3 Accuracy: 7.8431372549019605%
k=1, Overall Classification Accuracy: 34.66666666666667%
k=1, F1 Score (macro avg): 26.64%
k=2, Class 1 Accuracy: 72.41379310344827%
k=2, Class 2 Accuracy: 14.634146341463413%
k=2, Class 3 Accuracy: 7.8431372549019605%
k=2, Overall Classification Accuracy: 34.66666666666667%
k=2, F1 Score (macro avg): 26.64%
k=3, Class 1 Accuracy: 77.58620689655173%
k=3, Class 2 Accuracy: 12.195121951219512%
k=3, Class 3 Accuracy: 5.88235294117647%
k=3, Overall Classification Accuracy: 35.333333333333336%
k=3, F1 Score (macro avg): 25.56%
k=4, Class 1 Accuracy: 79.3103448275862%
k=4, Class 2 Accuracy: 12.195121951219512%
k=4, Class 3 Accuracy: 5.88235294117647%
k=4, Overall Classification Accuracy: 36.0%
k=4, F1 Score (macro avg): 25.89%
k=